## LAB7.2 — runaway, then cap  
Implements: **LAB7.2 (source V7.2)**

**Learning objective:** see an uncapped loop run forever, then halt it with `enforce_caps`. Uses a mock loop, so it's deterministic and free.

In [ ]:
import os, sys
# Prefer your working copy (my_agent/, from `python setup_module.py N`);
# fall back to the reference framework in src/.
for _p in ('../my_agent', '../src'):
    if os.path.isdir(os.path.join(_p, 'agent')):
        sys.path.insert(0, os.path.abspath(_p)); break

In [ ]:
from agent.loop import enforce_caps

# A mock 'agent step' that always wants to keep going and costs $0.10 each.
def mock_step():
    return {'wants_more': True, 'cost': 0.10}

In [ ]:
# The RUNAWAY (bounded here only by range() so the notebook returns!).
steps, cost = 0, 0.0
for _ in range(1000):
    s = mock_step(); steps += 1; cost += s['cost']
    if not s['wants_more']:
        break
print(f'runaway ran {steps} steps, ${cost:.2f} — it never stops on its own')

In [ ]:
steps, cost = 0, 0.0
for _ in range(1000):
    s = mock_step(); steps += 1; cost += s['cost']
    reason = enforce_caps(steps, cost, max_steps=5, max_cost_usd=0.30)
    if reason:
        print('HALTED:', reason); break

In [ ]:
# cost cap is checked after step cap in enforce_caps, so at (5, 0.50) the STEP cap fires first.
print(enforce_caps(5, 0.50, max_steps=5, max_cost_usd=0.20))      # max_steps
print(enforce_caps(3, 0.50, max_steps=5, max_cost_usd=0.20))      # max_cost

### 🔎 Eyeball your output
The runaway prints ~1000 steps. The capped loop should print `HALTED: max_cost ($0.3) reached` at step 3 (3 × $0.10 = $0.30) — the cost cap bites before the step cap here. That's the safety net that keeps a stuck agent from burning your budget.